# Structure Match — Database Field Exploration

Goal: for each data source, fetch one sample record and print every available field and its value.
Use Si (silicon) as the test structure throughout.

Sources covered:
1. Materials Cloud — OPTIMADE (`https://optimade.materialscloud.org`)
2. NOMAD — OPTIMADE (`https://nomad-lab.eu/prod/v1/optimade`)
3. JARVIS — OPTIMADE (`https://jarvis.nist.gov/optimade`)
4. Materials Project — `mp-api` (requires API key + `pip install mp-api`)
5. JARVIS — figshare bulk download (fallback; shows all available fields offline)

Run the OPTIMADE cells first — they need no credentials.
For MP, set your API key in the designated cell.

In [1]:
import json
import pprint
import requests

---
## Helper: OPTIMADE query

Reusable function for any OPTIMADE-compliant endpoint.

In [2]:
def optimade_query(base_url: str, filter_str: str, page_limit: int = 1) -> list[dict]:
    """Return raw `data` entries from an OPTIMADE /structures endpoint."""
    url = base_url.rstrip("/") + "/v1/structures"
    params = {"filter": filter_str, "page_limit": page_limit}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("data", [])


def print_fields(entry: dict, label: str = "") -> None:
    """Pretty-print the id, type, and all attribute keys + values of one OPTIMADE entry."""
    if label:
        print(f"=== {label} ===")
    print(f"id   : {entry.get('id')}")
    print(f"type : {entry.get('type')}")
    attrs = entry.get("attributes", {})
    print(f"\nAll attribute keys ({len(attrs)}):")
    for k in sorted(attrs):
        v = attrs[k]
        # truncate long lists
        if isinstance(v, list) and len(v) > 4:
            print(f"  {k}: [{v[0]}, {v[1]}, ... ({len(v)} items)]")
        elif isinstance(v, dict) and len(str(v)) > 120:
            print(f"  {k}: {{...}}")
        else:
            print(f"  {k}: {v}")

---
## 1. Materials Cloud (OPTIMADE)

In [3]:
MC_BASE = "https://optimade.materialscloud.org"

# filter: silicon, 2 atoms per cell (diamond cubic unit cell)
entries = optimade_query(MC_BASE, 'chemical_formula_reduced = "Si" AND nsites = 2')

if entries:
    print_fields(entries[0], label="Materials Cloud")
else:
    print("No results returned.")

HTTPError: 404 Client Error: Not Found for url: https://optimade.materialscloud.org/v1/structures?filter=chemical_formula_reduced+%3D+%22Si%22+AND+nsites+%3D+2&page_limit=1

In [4]:
# Check what provider-specific extensions exist (links, info)
r = requests.get(MC_BASE + "/v1/info", timeout=10)
info = r.json()
print("OPTIMADE version:", info.get("data", {}).get("attributes", {}).get("api_version"))
print("Available endpoints:", list(info.get("data", {}).get("attributes", {}).get("available_api_versions", {}).keys()))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

---
## 2. NOMAD (OPTIMADE)

In [5]:
NOMAD_BASE = "https://nomad-lab.eu/prod/v1/optimade"

entries = optimade_query(NOMAD_BASE, 'chemical_formula_reduced = "Si" AND nsites = 2')

if entries:
    print_fields(entries[0], label="NOMAD")
else:
    print("No results returned.")

=== NOMAD ===
id   : -F97NVxjGWo0nKZtQvsKPqfReI-a
type : structures

All attribute keys (459):
  _nmd_archive_url: https://nomad-lab.eu/prod/v1/api/archive/9FVTPztzTdGNVSEbc4QbXg/-F97NVxjGWo0nKZtQvsKPqfReI-a
  _nmd_calc_id: None
  _nmd_comment: None
  _nmd_description: None
  _nmd_description__suggestion: None
  _nmd_domain: None
  _nmd_entry_create_time: None
  _nmd_entry_id: None
  _nmd_entry_name: None
  _nmd_entry_name__suggestion: None
  _nmd_entry_page_url: https://nomad-lab.eu/prod/v1/gui/entry/id/9FVTPztzTdGNVSEbc4QbXg/-F97NVxjGWo0nKZtQvsKPqfReI-a
  _nmd_entry_type: None
  _nmd_external_db: None
  _nmd_external_id: None
  _nmd_files: None
  _nmd_last_processing_time: None
  _nmd_mainfile: None
  _nmd_mainfile__suggestion: None
  _nmd_mainfile_key: None
  _nmd_n_quantities: None
  _nmd_nomad_commit: None
  _nmd_nomad_distro_commit_url: None
  _nmd_nomad_version: None
  _nmd_origin: None
  _nmd_parser_name: None
  _nmd_pid: None
  _nmd_processed: None
  _nmd_processing_errors: No

In [6]:
# NOMAD has its own non-OPTIMADE API — check if more fields are available there
# by fetching the raw entry using the entry_id returned above
if entries:
    entry_id = entries[0]["id"]
    print("entry_id:", entry_id)
    # NOMAD own API (may have more fields like energy, band gap)
    # entry_id format for NOMAD own API may differ — just print it for reference
    print("\nTo query NOMAD own API, visit:")
    print(f"  https://nomad-lab.eu/prod/v1/api/v1/entries/{entry_id}")

entry_id: -F97NVxjGWo0nKZtQvsKPqfReI-a

To query NOMAD own API, visit:
  https://nomad-lab.eu/prod/v1/api/v1/entries/-F97NVxjGWo0nKZtQvsKPqfReI-a


---
## 3. JARVIS (OPTIMADE endpoint — test if available)

In [7]:
JARVIS_OPTIMADE_BASE = "https://jarvis.nist.gov/optimade"

try:
    entries = optimade_query(JARVIS_OPTIMADE_BASE, 'chemical_formula_reduced = "Si" AND nsites = 2')
    if entries:
        print_fields(entries[0], label="JARVIS OPTIMADE")
    else:
        print("No results (endpoint may exist but returned empty).")
except Exception as e:
    print(f"JARVIS OPTIMADE endpoint not available: {e}")
    print("Fall back to jarvis-tools bulk download (see Section 5).")

JARVIS OPTIMADE endpoint not available: 404 Client Error: Not Found for url: https://jarvis.nist.gov/optimade/v1/structures?filter=chemical_formula_reduced+%3D+%22Si%22+AND+nsites+%3D+2&page_limit=1
Fall back to jarvis-tools bulk download (see Section 5).


---
## 4. Materials Project (mp-api)

Requires:
```
uv add mp-api   # or: pip install mp-api
```
Get your API key at https://next-gen.materialsproject.org/api

In [9]:
MP_API_KEY = "pXeWrwRQb5sFVnwjzJcmr742DUWHa3oD"  # replace before running

try:
    from mp_api.client import MPRester

    with MPRester(MP_API_KEY) as mpr:
        # Fetch one Si entry with all summary fields
        docs = mpr.materials.summary.search(
            formula="Si",
            num_sites=(2, 2),
        )

    if docs:
        doc = docs[0]
        print(f"material_id : {doc.material_id}")
        print(f"\nAll fields on SummaryDoc:")
        for field, value in doc.__dict__.items():
            if field.startswith("_"):
                continue
            if isinstance(value, list) and len(value) > 4:
                print(f"  {field}: [{value[0]}, ... ({len(value)} items)]")
            else:
                print(f"  {field}: {value}")
    else:
        print("No results.")
except ImportError:
    print("mp-api not installed. Run: uv add mp-api")

/opt/anaconda3/envs/goldilocks-explore/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Retrieving SummaryDoc documents: 100%|██████████| 3/3 [00:00<00:00, 56679.78it/s]

material_id : mp-92

All fields on SummaryDoc:
  builder_meta: emmet_version='0.84.3rc4' pymatgen_version='2024.11.13' run_id='32bfb79c-5ce0-41ab-ab69-69ba9fb96205' batch_id=None database_version='2025.09.25' build_date=datetime.datetime(2024, 11, 21, 22, 29, 31, 737000, tzinfo=datetime.timezone.utc) license='BY-C'
  nsites: 2
  elements: [Element Si]
  nelements: 1
  composition: Si2
  composition_reduced: Si1
  formula_pretty: Si
  formula_anonymous: A
  chemsys: Si
  volume: 29.899999100890987
  density: 3.119536545645218
  density_atomic: 14.949999550445494
  symmetry: crystal_system=<CrystalSystem.tet: 'Tetragonal'> symbol='I4_1/amd' hall=None number=141 point_group='4/mmm' symprec=0.1 angle_tolerance=5.0 version='2.5.0'
  material_id: mp-92
  deprecated: False
  deprecation_reasons: None
  last_updated: 2025-09-23 22:39:42.522000+00:00
  origins: [PropertyOrigin(name='structure', task_id=MPID(mp-1524873), last_updated=datetime.datetime(2020, 5, 8, 22, 24, 44, 964000, tzinfo=datet

In [10]:
# Alternative: list all available query fields without fetching data
try:
    from mp_api.client import MPRester
    with MPRester(MP_API_KEY) as mpr:
        fields = mpr.materials.summary.available_fields
        print(f"{len(fields)} available fields:")
        for f in sorted(fields):
            print(" ", f)
except ImportError:
    print("mp-api not installed.")

69 available fields:
  band_gap
  bandstructure
  builder_meta
  bulk_modulus
  cbm
  chemsys
  composition
  composition_reduced
  database_IDs
  decomposes_to
  density
  density_atomic
  deprecated
  deprecation_reasons
  dos
  dos_energy_down
  dos_energy_up
  e_electronic
  e_ij_max
  e_ionic
  e_total
  efermi
  elements
  energy_above_hull
  energy_per_atom
  equilibrium_reaction_energy_per_atom
  es_source_calc_id
  formation_energy_per_atom
  formula_anonymous
  formula_pretty
  grain_boundaries
  has_props
  has_reconstructed
  homogeneous_poisson
  is_gap_direct
  is_magnetic
  is_metal
  is_stable
  last_updated
  material_id
  n
  nelements
  nsites
  num_magnetic_sites
  num_unique_magnetic_sites
  ordering
  origins
  possible_species
  property_name
  shape_factor
  shear_modulus
  structure
  surface_anisotropy
  symmetry
  task_ids
  theoretical
  total_magnetization
  total_magnetization_normalized_formula_units
  total_magnetization_normalized_vol
  types_of_magneti

---
## 5. JARVIS — figshare bulk download (offline fallback)

Downloads the full `dft_3d` dataset (~76k entries) from Figshare on first run.
Cached locally after that. Shows all available fields without network queries.

Requires: `uv add jarvis-tools`

In [11]:
try:
    from jarvis.db.figshare import data as jarvis_data

    dft_3d = jarvis_data("dft_3d")  # downloads ~200 MB on first run, cached after
    sample = dft_3d[0]

    print(f"Total entries: {len(dft_3d)}")
    print(f"\nAll fields in one record ({len(sample)} keys):")
    for k, v in sorted(sample.items()):
        if isinstance(v, dict) and len(str(v)) > 120:
            print(f"  {k}: {{...}}")
        elif isinstance(v, list) and len(v) > 4:
            print(f"  {k}: [{v[0]}, ... ({len(v)} items)]")
        else:
            print(f"  {k}: {v}")
except ImportError:
    print("jarvis-tools not installed. Run: uv add jarvis-tools")

Obtaining 3D dataset 76k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699


100%|██████████| 40.8M/40.8M [00:00<00:00, 48.1MiB/s]


Loading the zipfile...
Loading completed.
Total entries: 75993

All fields in one record (64 keys):
  Tc_supercon: na
  atoms: {...}
  avg_elec_mass: na
  avg_hole_mass: na
  bulk_modulus_kv: na
  crys: tetragonal
  density: 5.956
  dfpt_piezo_max_dielectric: na
  dfpt_piezo_max_dielectric_electronic: na
  dfpt_piezo_max_dielectric_ionic: na
  dfpt_piezo_max_dij: na
  dfpt_piezo_max_eij: na
  dimensionality: 3D-bulk
  effective_masses_300K: {'p': 'na', 'n': 'na'}
  efg: []
  ehull: 0.0423
  elastic_tensor: na
  encut: 650
  epsx: 76.23
  epsy: 76.23
  epsz: 54.0402
  exfoliation_energy: na
  formation_energy_peratom: -0.42762
  formula: TiCuSiAs
  func: OptB88vdW
  hse_gap: na
  icsd: 
  jid: JVASP-90856
  kpoint_length_unit: 60
  magmom_oszicar: 0.0
  magmom_outcar: 0.0
  max_efg: na
  max_ir_mode: na
  maxdiff_bz: na
  maxdiff_mesh: na
  mbj_bandgap: na
  mepsx: na
  mepsy: na
  mepsz: na
  min_ir_mode: na
  modes: na
  n-Seebeck: na
  n-powerfact: na
  nat: 8
  ncond: na
  nkappa: n

In [12]:
# Filter to Si entries and print a few records
try:
    si_entries = [e for e in dft_3d if e.get("formula") == "Si"]
    print(f"Si entries: {len(si_entries)}")
    for entry in si_entries[:3]:
        print("---")
        pprint.pprint({k: v for k, v in entry.items() if k != "atoms"})
except NameError:
    print("dft_3d not loaded — run the cell above first.")

Si entries: 32
---
{'Tc_supercon': 'na',
 'avg_elec_mass': 0.0,
 'avg_hole_mass': 0.0,
 'bulk_modulus_kv': 81.6,
 'crys': 'cubic',
 'density': 2.469,
 'dfpt_piezo_max_dielectric': 'na',
 'dfpt_piezo_max_dielectric_electronic': 'na',
 'dfpt_piezo_max_dielectric_ionic': 'na',
 'dfpt_piezo_max_dij': 'na',
 'dfpt_piezo_max_eij': 'na',
 'dimensionality': '3D-bulk',
 'effective_masses_300K': {'n': [0.0, 0.0, 0.0], 'p': [0.0, 0.0, 0.0]},
 'efg': [],
 'ehull': 0.0,
 'elastic_tensor': [[170.2, 37.3, 37.3, -0.0, -0.0, -0.0],
                    [37.3, 170.2, 37.3, 0.0, 0.0, -0.0],
                    [37.3, 37.3, 170.2, 0.0, 0.0, -0.0],
                    [-0.0, 0.0, 0.0, 64.6, -0.0, 0.0],
                    [-0.0, -0.0, 0.0, -0.0, 64.6, 0.0],
                    [-0.0, -0.0, -0.0, 0.0, 0.0, 64.6]],
 'encut': 500,
 'epsx': 25.264,
 'epsy': 25.2641,
 'epsz': 25.2641,
 'exfoliation_energy': 'na',
 'formation_energy_peratom': 0.16663,
 'formula': 'Si',
 'func': 'OptB88vdW',
 'hse_gap': 'na',
 'ic

---
## Summary

After running the cells above, fill in this table:

| Field | Materials Cloud | NOMAD | JARVIS | Materials Project |
|---|---|---|---|---|
| material ID | | | | |
| formula | | | | |
| space group | | | | |
| formation energy | | | | |
| band gap | | | | |
| structure (fractional coords) | | | | |

Mark each cell: **yes** / **no** / **key name if different**.

---
## 6. Link URL Verification

Confirm the web page URL pattern for each source.

### NOMAD — already confirmed

The OPTIMADE response includes `_nmd_entry_page_url` directly in `attributes`.
No need to construct — just read the field.

Example: `https://nomad-lab.eu/prod/v1/gui/entry/id/9FVTPztzTdGNVSEbc4QbXg/-F97NVxjGWo0nKZtQvsKPqfReI-a`

### Materials Cloud — query correct per-dataset endpoint

In [ ]:
MC_MC3D = "https://optimade.materialscloud.org/main/mc3d-pbesol-v2"

try:
    mc_entries = optimade_query(MC_MC3D, 'chemical_formula_reduced = "Si" AND nsites = 2')
    if mc_entries:
        entry = mc_entries[0]
        print_fields(entry, label="Materials Cloud mc3d-pbesol-v2")
        print("
--- links object ---")
        print(json.dumps(entry.get("links", {}), indent=2))
        print("
--- relationships object ---")
        print(json.dumps(entry.get("relationships", {}), indent=2))
    else:
        print("No results returned.")
except Exception as e:
    print(f"Error: {e}")


In [13]:
# Check if MC OPTIMADE entry has a links.self pointing to a web page
if mc_entries:
    entry = mc_entries[0]
    entry_id = entry["id"]
    print("entry id:", entry_id)
    # Try constructing the MC web explore URL
    mc_web_url = f"https://www.materialscloud.org/explore/mc3d/details/{entry_id}"
    print("Candidate web URL:", mc_web_url)
    # Check if that URL exists
    r = requests.get(mc_web_url, timeout=10, allow_redirects=True)
    print("HTTP status:", r.status_code)
    # Also try the OPTIMADE self link
    self_link = f"{MC_MC3D}/v1/structures/{entry_id}"
    print("OPTIMADE self link:", self_link)


NameError: name 'mc_entries' is not defined

### JARVIS — check web URL from jid

In [14]:
# JARVIS entries have jid e.g. JVASP-14831
# Test candidate URL patterns
test_jid = "JVASP-14831"  # Si tetragonal, known entry

candidates = [
    f"https://jarvis.nist.gov/jarvisdft/{test_jid}",
    f"https://www.ctcms.nist.gov/~knc6/jvasp.html?jid={test_jid}",
    f"https://jarvis.nist.gov/?jid={test_jid}",
]

for url in candidates:
    try:
        r = requests.get(url, timeout=10, allow_redirects=True)
        print(f"{r.status_code}  {r.url}")
        print(f"  final url: {r.url}")
        print(f"  content snippet: {r.text[:200].strip()}")
        print()
    except Exception as e:
        print(f"FAIL {url}: {e}")


404  https://jarvis.nist.gov/jarvisdft/JVASP-14831
  final url: https://jarvis.nist.gov/jarvisdft/JVASP-14831
  content snippet: <!doctype html>
<html lang="en">
<head>
  <title>Not Found</title>
</head>
<body>
  <h1>Not Found</h1><p>The requested resource was not found on this server.</p>
<script>(function(){function c(){var

404  https://www.ctcms.nist.gov/~knc6/jvasp.html?jid=JVASP-14831
  final url: https://www.ctcms.nist.gov/~knc6/jvasp.html?jid=JVASP-14831
  content snippet: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>404 Not Found</title>
</head><body>
<h1>Not Found</h1>
<p>The requested URL was not found on this server.</p>
<script>(function()

200  https://jarvis.nist.gov/?jid=JVASP-14831
  final url: https://jarvis.nist.gov/?jid=JVASP-14831
  content snippet: <!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1"/>
    
    <meta name="author" content="NIST" />
    <